# Classificação de Pesquisadores com Gemini 2.5 Flash Lite

Este notebook prepara a base de dados de auxílios da FAPESP, classifica os pesquisadores com apoio do modelo Gemini e gera arquivos separados para clientes e não clientes.

## 1. Instalar dependências

Execute a célula abaixo caso o ambiente ainda não tenha as bibliotecas necessárias.

In [ ]:
!pip install -q pandas tqdm google-generativeai


## 2. Imports e configuração da API

Defina a variável de ambiente `GOOGLE_API_KEY` antes de continuar (por exemplo, `export GOOGLE_API_KEY="sua_chave"`).

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import Dict, List

import pandas as pd
from tqdm.auto import tqdm
import google.generativeai as genai

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Defina a variável de ambiente GOOGLE_API_KEY antes de continuar.")

genai.configure(api_key=GOOGLE_API_KEY)


## 3. Parâmetros principais

Ajuste os caminhos e hiperparâmetros conforme necessário.

In [ ]:
MODEL_NAME = "gemini-2.5-flash-lite"
INPUT_CSV_PATH = "/content/auxilios_em_andamento.csv"
OUTPUT_NAME = "teste1"
OUTPUT_DIR = Path("/content")
OUTPUT_CSV_PATH = OUTPUT_DIR / f"{OUTPUT_NAME}.csv"
BATCH_SIZE = 15
MAX_RETRIES = 3
SLEEP_BETWEEN_RETRIES = 5  # segundos
TEMPERATURE = 0.0


## 4. Heurística positiva

Lista de sinais que caracterizam um potencial cliente. Ela é usada na instrução do modelo e como *fallback* automático caso a chamada ao LLM falhe.

In [ ]:
POSITIVE_SIGNALS = [
    "síntese de gene",
    "gene synthesis",
    "produção de proteína",
    "protein expression",
    "antígeno",
    "antigen",
    "ensaios elisa",
    "elisa",
    "biologia molecular",
    "biologia celular",
    "bioquímica",
    "anticorpo",
    "clonagem",
    "purificação de proteína",
    "expressão recombinante",
    "sequenciamento",
    "vetor de expressão",
    "rna mensageiro",
    "mrna",
    "engenharia genética",
    "produção de peptídeo"
]

POSITIVE_KEYWORDS = [signal.lower() for signal in POSITIVE_SIGNALS]


## 5. Carregar e preparar o DataFrame base

A função abaixo seleciona apenas as colunas necessárias (`Nome` e `Resumo (Português)`) e remove linhas vazias ou duplicadas.

In [ ]:
def load_and_prepare_dataframe(csv_path: str) -> pd.DataFrame:
    raw_df = pd.read_csv(csv_path)

    name_candidates = [
        "Nome do Beneficiário",
        "Beneficiário",
        "Pesquisador Responsável",
        "Pesquisador",
    ]
    summary_candidates = [
        "Resumo (Português)",
        "Resumo em Português",
        "Resumo",
    ]

    name_col = next((col for col in name_candidates if col in raw_df.columns), None)
    summary_col = next((col for col in summary_candidates if col in raw_df.columns), None)

    if name_col is None:
        raise KeyError("Nenhuma coluna de nome encontrada no CSV. Revise os cabeçalhos.")
    if summary_col is None:
        raise KeyError("Nenhuma coluna de resumo em português encontrada no CSV.")

    df = (
        raw_df[[name_col, summary_col]]
        .rename(columns={name_col: "nome", summary_col: "resumo"})
        .dropna(subset=["nome", "resumo"])
    )

    df["nome"] = df["nome"].astype(str).str.strip()
    df["resumo"] = df["resumo"].astype(str).str.strip()

    df = df[(df["nome"] != "") & (df["resumo"] != "")]
    df = df.drop_duplicates(subset=["nome", "resumo"]).reset_index(drop=True)
    df["entrada_id"] = df.index
    return df

base_df = load_and_prepare_dataframe(str(INPUT_CSV_PATH))
print(f"Entradas carregadas: {len(base_df)}")
base_df.head()


## 6. Preparar o prompt e funções de classificação

O modelo recebe lotes de resumos e retorna uma lista JSON com a decisão para cada pesquisador.

In [ ]:
model = genai.GenerativeModel(model_name=MODEL_NAME)

def build_batch_prompt(batch_rows: List[Dict[str, str]]) -> str:
    header = """Você é um assistente que classifica pesquisadores como potenciais clientes.
Considere apenas os sinais a seguir como positivos:
- Síntese de gene
- Produção ou expressão de proteína
- Antígenos e anticorpos
- Ensaios ELISA ou imunoensaios semelhantes
- Biologia molecular, biologia celular ou bioquímica experimental

Instruções:
1. Para cada resumo, responda se ele descreve atividades alinhadas às categorias positivas.
2. Se houver aderência clara a qualquer uma das categorias, marque como "CLIENTE".
3. Caso contrário, marque como "NAO_CLIENTE".
4. Justifique brevemente com base em termos presentes no resumo (sem inventar informações).
5. Responda apenas em JSON com a estrutura:
[
  {"entrada_id": <numero>, "nome": "...", "classificacao": "CLIENTE" ou "NAO_CLIENTE", "justificativa": "..."}
]
"""
    parts = [header, "Entradas:"]
    for row in batch_rows:
        parts.append("###")
        parts.append(f"entrada_id: {row['entrada_id']}")
        parts.append(f"nome: {row['nome']}")
        parts.append(f"resumo: {row['resumo']}")
    parts.append("### Fim das entradas ###")
    parts.append("Reforço: responda apenas com o JSON descrito, sem texto adicional.")
    return "
".join(parts)


def classify_batch(batch_rows: List[Dict[str, str]]):
    prompt = build_batch_prompt(batch_rows)
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=TEMPERATURE,
                    max_output_tokens=2048,
                ),
            )
            text = response.text.strip()
            parsed = json.loads(text)

            if not isinstance(parsed, list):
                raise ValueError("Resposta do modelo não é uma lista JSON.")

            normalized = []
            entrada_ids = {row["entrada_id"] for row in batch_rows}

            for item in parsed:
                normalized.append(
                    {
                        "entrada_id": int(item["entrada_id"]),
                        "nome": str(item["nome"]).strip(),
                        "classificacao": str(item["classificacao"]).strip().upper(),
                        "justificativa": str(item.get("justificativa", "")).strip(),
                        "modo_classificacao": "llm",
                        "raw_response": text,
                    }
                )

            ids_retornados = {row["entrada_id"] for row in normalized}
            if entrada_ids != ids_retornados:
                raise ValueError("Resposta do modelo não cobriu todas as entradas.")

            return normalized
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(SLEEP_BETWEEN_RETRIES)
            else:
                print(f"LLM falhou após {MAX_RETRIES} tentativas: {last_error}")

    # Fallback baseado em palavras-chave
    fallback_results = []
    for row in batch_rows:
        resumo_lower = row["resumo"].lower()
        is_client = any(keyword in resumo_lower for keyword in POSITIVE_KEYWORDS)
        fallback_results.append(
            {
                "entrada_id": row["entrada_id"],
                "nome": row["nome"],
                "classificacao": "CLIENTE" if is_client else "NAO_CLIENTE",
                "justificativa": "Classificação por heurística de palavras-chave." if is_client else "Sem sinais positivos identificados (fallback).",
                "modo_classificacao": "fallback_keyword",
                "raw_response": None,
            }
        )
    return fallback_results


## 7. Executar a classificação

Processa o DataFrame em lotes e agrega os resultados.

In [ ]:
results = []
rows = base_df.to_dict("records")
for start in tqdm(range(0, len(rows), BATCH_SIZE)):
    batch = rows[start : start + BATCH_SIZE]
    batch_results = classify_batch(batch)
    results.extend(batch_results)

results_df = pd.DataFrame(results)
final_df = base_df.merge(results_df, on=["entrada_id", "nome"], how="left")
final_df.head()


## 8. Exportar relatórios

Gera um arquivo consolidado com todas as classificações e dois arquivos auxiliares separando clientes e não clientes.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Arquivo principal salvo em: {OUTPUT_CSV_PATH}")

clientes_df = final_df[final_df["classificacao"] == "CLIENTE"].copy()
nao_clientes_df = final_df[final_df["classificacao"] == "NAO_CLIENTE"].copy()

clientes_path = OUTPUT_CSV_PATH.with_name(f"{OUTPUT_CSV_PATH.stem}_clientes.csv")
nao_clientes_path = OUTPUT_CSV_PATH.with_name(f"{OUTPUT_CSV_PATH.stem}_nao_clientes.csv")

clientes_df.to_csv(clientes_path, index=False)
nao_clientes_df.to_csv(nao_clientes_path, index=False)

print(f"Clientes: {len(clientes_df)} linhas → {clientes_path}")
print(f"Não clientes: {len(nao_clientes_df)} linhas → {nao_clientes_path}")


## 9. Visualizar as primeiras classificações

In [ ]:
final_df[['nome', 'classificacao', 'justificativa', 'modo_classificacao']].head()
